# Lecture 1 · Meet Amazon Bedrock AgentCore

**Course:** *AI Fundamentals for Beginners: Learn LLM, Agentic AI & MCP*

Over the next three notebooks you will build a real **AIOps agent** — an assistant you can ask *"what's been happening in my AWS account?"* or *"stop the staging server"* in plain English, and that goes and does it. It will read your actual CloudTrail history, list your actual EC2 instances, and refuse to do anything dangerous.

You'll build it on **Amazon Bedrock AgentCore**, AWS's platform for running AI agents in production.

### What you will do in this notebook

1. Understand what an *agent* is, and what AgentCore gives you
2. Install the Python packages
3. Confirm your AWS credentials work
4. Confirm you have access to a Claude model on Bedrock
5. Save the settings the next two notebooks will read

**Time:** about 15 minutes. **Cost:** a few cents — one small model call.

> **The three notebooks**
> `01` (this one) — concepts and setup
> `02` — build the agent and run it on your laptop
> `03` — deploy it to AgentCore Runtime and operate it

---
## 1. What is an agent, really?

Strip away the hype and an agent is a small, specific idea:

> **An agent is a language model that has been given a set of functions it may ask you to run, looped until it has an answer.**

That's it. The model can't run code and can't reach AWS. What it *can* do is say *"I'd like you to call `list_instances(state='running')`"* — and your program decides whether to honour that, runs the function, hands back the result, and asks the model what it wants to do next. Round and round until the model stops asking and just answers.

Three consequences follow, and they shape everything in this course:

**The model picks the tools.** You never write `if user_asks_about_instances: list_instances()`. You describe your tools well and the model matches them to the question. Notebook 02 shows what "describe well" means in practice.

**Your code is the gatekeeper.** Every action passes through a function you wrote. The gap between *the model asking* and *the thing happening* is where all safety lives.

**Results accumulate.** Each tool result gets appended to the conversation and re-sent on every later turn. Big results make agents slow, expensive, and vague. Small results keep them sharp.

We'll watch this loop actually turn in notebook 02. For now, just hold the shape of it.

---
## 2. What AgentCore gives you

You could write that loop yourself in an afternoon — and on your laptop, that's fine. Then you try to make it something your team can use, and a different set of problems shows up:

- Where does it *run*, so it isn't tied to your laptop?
- Two people ask it something at once. Do they see each other's conversation?
- It runs with *your* AWS credentials. Should it be able to do everything you can?
- It gave a strange answer yesterday. Which tool did it call, and what came back?
- It should remember that Priya always works in `eu-west-1`. Where does that live?

**AgentCore is AWS's set of managed answers to those questions.** Not one service — a menu you pick from:

| Service | What it does | Used in this course? |
| --- | --- | --- |
| **Runtime** | Hosts your agent. Serverless, session-isolated, up to 8-hour runs. | ✅ Notebook 03 |
| **Observability** | Traces, spans, and logs for every agent turn, in CloudWatch. | ✅ Notebook 03 |
| **Memory** | Short-term (this conversation) and long-term (across sessions) memory. | ✖ Out of scope |
| **Gateway** | Turns existing APIs and Lambda functions into agent tools via MCP. | ✖ Out of scope |
| **Identity** | OAuth and API-key credentials so an agent can act as a user. | ✖ Not needed here |
| **Browser** | A managed, sandboxed browser the agent can drive. | ✖ |
| **Code Interpreter** | A sandbox for running code the agent writes. | ✖ |

You do not need to learn all of these. You need **Runtime**. The rest are there when a problem asks for them, and knowing they exist is enough for now.

Only **Runtime** and **Observability** are used in this course. The others are named once here so you recognise them in the docs; notebook 03 closes with a short word on Memory and Gateway, and that is as far as we go — each is a course of its own.

**AgentCore is deliberately framework-agnostic.** You write the agent loop with a framework you choose — we'll use **Strands Agents**, but LangGraph, Google ADK, and OpenAI Agents all deploy the same way. AgentCore hosts and operates your agent; it does not dictate how you write it.

### The shape of what you'll build

```
  You ──▶ POST /invocations ──▶ ┌────────────────────────────────┐
                                │  AgentCore Runtime             │
                                │  (session-isolated microVM)    │
                                │  ┌──────────────────────────┐  │
                                │  │ main.py                  │  │  ← you write this
                                │  │  Strands Agent loop      │  │
                                │  │  + SYSTEM_PROMPT         │  │
                                │  │  + @tool functions ──────┼──┼──▶ CloudTrail / EC2
                                │  └──────────────────────────┘  │
                                └────────────────────────────────┘
                                    │
                                    └─▶ Observability (traces, spans, logs)
```

Notice how much of that diagram is **yours**. The prompt, the tools, the loop — all ordinary Python in a file you can open. AgentCore supplies the box around it: the hosting, the isolation, the tracing.

The single most useful property of this design: **the same `main.py` runs on your laptop and in the cloud.** You will spend notebook 02 iterating locally in seconds, and notebook 03 shipping that exact file unchanged.

### What the agent will actually do

An **AIOps** agent — AI for IT operations. Two jobs:

**Monitoring (read-only, always safe)** — query CloudTrail to answer *"what happened recently?"*, *"what did user alice do today?"*, *"show me recent EC2 activity"*.

**Management (changes real infrastructure)** — start, stop, and reboot EC2 instances.

That second job is why this is a good teaching project rather than a toy chatbot. Once an LLM can stop servers, "it usually behaves correctly" stops being good enough, and you have to think properly about safety. Notebook 02 builds three independent layers of it.

---
## 3. Before you run anything

You need:

- **An AWS account** where you are allowed to create IAM roles.
- **Git**, to clone this repo.
- **AWS CLI v2**, to configure IAM credentials by running `aws configure sso` (recommended) or
  `aws configure`, then verify with `aws sts get-caller-identity`. The notebooks
  print `aws ...` commands for you to run, and notebook 03 needs it to deploy.
- **Bedrock model access.** Serverless models are now enabled by default in all commercial regions — the old *Model access* page is retired, and there is nothing to request per model. Anthropic is the one exception: AWS requires a **one-time use-case form per account** before your first Claude call. Open any Claude model in the [Bedrock console](https://console.aws.amazon.com/bedrock) model catalog and submit it, or call the `PutUseCaseForModelAccess` API. Access is granted immediately.
- **CloudTrail enabled** — it is on by default for management events, so you almost certainly have it.
- **Python 3.10+** and Jupyter.
- *(Notebook 03 only)* **Node.js 20+**, for the AgentCore CLI.

> **Don't have Git, the AWS CLI, Python, or Node.js yet?** The repo README has
> copy-paste install commands for macOS, Windows, and Linux, direct download links
> for the official installers if you prefer clicking through them, and how to sign
> the CLI in to AWS:
> [Installing Git, the AWS CLI, Python, and Node.js](../README.md#installing-git-the-aws-cli-python-and-nodejs)
> · [Download the installers directly](../README.md#download-the-installers-directly).
> Do that first, then come back here — section 5 below checks your credentials and
> will fail if the CLI is not configured.

**An EC2 instance is optional.** The agent has a dry-run mode, so you can do every exercise without one.

### Getting the code and starting Jupyter

```bash
git clone https://github.com/himanshurgit/aiops-agent-awsagentcore.git
cd aiops-agent-awsagentcore
python3 -m venv .venv && source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

On Windows PowerShell the activate line is `.venv\Scripts\Activate.ps1`.

### A word on cost

Running all three notebooks costs roughly **$0.50–$2.00**: a few hundred thousand Bedrock tokens, plus AgentCore Runtime billed per second while a request is in flight. Idle deployed agents cost nothing. Notebook 03 ends with a cleanup step — please run it.

---
## 4. Install the packages

Two packages do the work:

- **`strands-agents`** — the agent framework. It owns the loop from section 1: send the conversation to the model, notice when the model asks for a tool, run that tool, feed the result back, repeat until the model is done.
- **`bedrock-agentcore`** — the AWS SDK for AgentCore. It gives you `BedrockAgentCoreApp`, the HTTP contract AgentCore Runtime speaks, plus clients for the other AgentCore services.

`boto3` you may already know — it's how our tools will talk to CloudTrail and EC2.

In [ ]:
%pip install -q --upgrade "strands-agents>=1.0.0" "bedrock-agentcore>=0.1.0" "boto3>=1.35.0"
print("Installed. If this was your first install, restart the kernel now:")
print("  Kernel -> Restart Kernel, then continue from the next cell.")

---
## 5. Check your AWS credentials

Before anything else, prove that this notebook can talk to AWS as the identity you expect. `sts:GetCallerIdentity` is the cheapest possible way to ask "who am I?" — every AWS identity is allowed to call it, and it changes nothing.

**Edit `REGION` below** if you work somewhere other than `us-east-1`. Use a region where Bedrock offers Claude models.

In [ ]:
import boto3
import botocore

REGION = "us-east-1"          # <-- change this if you work in another region

session = boto3.Session(region_name=REGION)

try:
    who = session.client("sts").get_caller_identity()
    print("Account :", who["Account"])
    print("Identity:", who["Arn"])
    print("Region  :", session.region_name)
    print("\nCredentials OK.")
except botocore.exceptions.NoCredentialsError:
    print("No AWS credentials found.")
    print("Fix: run `aws configure sso` (then `aws sso login`), or `aws configure`.")
except botocore.exceptions.ClientError as err:
    code_ = err.response["Error"]["Code"]
    print("Credentials were found but AWS rejected them:", code_)
    if "ExpiredToken" in code_:
        print("Fix: your SSO session expired. Run `aws sso login` and restart the kernel.")

> **If this failed:** the notebook inherits credentials from your shell environment. If you ran `aws sso login` *after* starting Jupyter, Jupyter did not see it — restart Jupyter, not just the kernel.

---
## 6. Find a model you can actually use

Two things can go wrong here, and beginners hit both:

**1. Access is automatic now — with one exception.** You may have read older guides telling you to visit a *Model access* page and request each model. **That page no longer exists.** Every serverless foundation model is enabled by default in commercial regions.

Amazon's own models — the **Nova** family this course defaults to — work immediately. Nothing to sign, no subscription.

**Anthropic's Claude models keep one requirement:** a single use-case form, submitted once per account (or once at the management account of an AWS Organization, which members inherit). Submit it by opening any Claude model in the Bedrock console's model catalog, or by calling `PutUseCaseForModelAccess`. Access is granted the moment you submit.

**If you tried a Claude model and got `AccessDeniedException`, that form is almost certainly why** — and it is the reason this course defaults to Nova.

One more thing with the same symptom: your first call to a *third-party* model quietly starts an AWS Marketplace subscription in the background, which needs `aws-marketplace:Subscribe` and a valid payment method. It can take a few minutes to settle, so a transient denial on a first Claude call is normal — wait and rerun. Amazon's own models skip this entirely.

**2. You used a base model ID where an inference profile ID was needed.** Most current Bedrock models are served through **cross-region inference profiles**. The ID looks like `us.amazon.nova-2-lite-v1:0` — note the `us.` prefix. The bare `amazon.nova-2-lite-v1:0` will fail with a `ValidationException` in most regions. The prefix tells Bedrock which pool of regions may serve your request:

- `us.` — routed within the US and Canada
- `eu.` — routed within the EU
- `global.` — routed anywhere, cheapest to get capacity, no data-residency guarantee

The cell below lists the Claude profiles your account can see.

In [ ]:
bedrock = session.client("bedrock")

try:
    profiles = bedrock.list_inference_profiles()["inferenceProfileSummaries"]
    wanted = ("nova", "claude")
    matches = sorted(
        (p for p in profiles
         if any(w in p["inferenceProfileId"].lower() for w in wanted)),
        key=lambda p: p["inferenceProfileId"],
    )
    if matches:
        print(f"Nova and Claude inference profiles available in {REGION}:\n")
        for p in matches:
            print(f"  {p['inferenceProfileId']:<50} {p['status']}")
        print(f"\n{len(matches)} found. Copy one into MODEL_ID in the next cell.")
    else:
        print("No Nova or Claude profiles offered in this region.")
        print("Fix: try us-east-1 or us-west-2, which carry the full lineup.")
except botocore.exceptions.ClientError as err:
    print("Could not list inference profiles:", err.response["Error"]["Code"])
    print("Your identity may be missing bedrock:ListInferenceProfiles.")

### Which one should you pick?

- **Amazon Nova Lite** — the default for this course. Fast, cheap, supports tool use, and needs no paperwork to enable.
- **Amazon Nova Pro / Premier** — more capable and more expensive. Worth switching to if the agent misreads multi-step questions.
- **Anthropic Claude** — very strong at tool use and what many production agents run on. Needs the one-time use-case form described above.

Everything in this course works with any of them. Pick one from the list **your own account** printed — hard-coded IDs go stale as new models ship.

In [ ]:
MODEL_ID = "us.amazon.nova-2-lite-v1:0"     # <-- paste your chosen ID here

runtime = session.client("bedrock-runtime")

try:
    response = runtime.converse(
        modelId=MODEL_ID,
        messages=[{"role": "user", "content": [{"text": "Reply with exactly: AgentCore setup OK"}]}],
        inferenceConfig={"maxTokens": 32},
    )
    print("Model replied:", response["output"]["message"]["content"][0]["text"])
    print("Token usage  :", response["usage"])
    print("\nYou have working model access. This is the last thing that had to be true.")
except botocore.exceptions.ClientError as err:
    code_ = err.response["Error"]["Code"]
    print(f"Model call failed: {code_}\n{err.response['Error']['Message']}\n")
    if code_ == "AccessDeniedException":
        print("-> Anthropic needs a one-time use-case form per account. Open any Claude")
        print("   model in the Bedrock console catalog and submit it, or call")
        print("   bedrock:PutUseCaseForModelAccess.")
        print("-> Already submitted? The first call starts a Marketplace subscription")
        print("   that can take a few minutes. Wait and rerun this cell.")
        print("-> Your identity needs aws-marketplace:Subscribe for that first call.")
    elif code_ == "ValidationException":
        print("-> The model ID is wrong for this region. Use an ID from the list above,")
        print("   including its us./eu./global. prefix.")
    elif code_ == "ThrottlingException":
        print("-> Rate limited. Wait a few seconds and rerun this cell.")

> **What is `converse`?** It's Bedrock's model-agnostic chat API — the same call shape works for Claude, Nova, Llama, and the rest. You will not call it directly again in this course; Strands calls it for you. We used it here purely as a one-line proof that your access works.

---
## 7. Save your settings

Notebooks 02 and 03 read this file, so you only choose your region and model once.

In [ ]:
import json
from pathlib import Path

settings = {"region": REGION, "model_id": MODEL_ID}
Path("course_settings.json").write_text(json.dumps(settings, indent=2))

print("Saved to course_settings.json:")
print(json.dumps(settings, indent=2))

---
## 8. Troubleshooting

| Symptom | Cause | Fix |
| --- | --- | --- |
| `NoCredentialsError` | Jupyter can't see your AWS config | `aws sso login`, then restart **Jupyter itself**, not just the kernel |
| `ExpiredTokenException` | SSO session timed out | `aws sso login`, restart Jupyter |
| `AccessDeniedException` on a **Claude** model | Anthropic use-case form not yet submitted | Open any Claude model in the Bedrock console catalog and submit the form — or just use a Nova model, which needs no form |
| `AccessDeniedException` on your very first call only | Marketplace subscription still settling | Wait a few minutes and retry. Needs `aws-marketplace:Subscribe` and a valid payment method |
| `ValidationException: model identifier is invalid` | Base ID used instead of a profile ID | Use the `us.`-prefixed ID from the list |
| `ThrottlingException` | Too many calls too fast | Wait, then retry. New accounts have low Bedrock quotas |
| `ModuleNotFoundError: strands` | Kernel started before the install | Restart the kernel and rerun from the install cell |
| Model list is empty | Bedrock unavailable in that region | Try `us-east-1` or `us-west-2` |

---
## Recap

- An **agent** is a model that can ask for functions to be run, looped until it has an answer. The model chooses; your code decides whether to comply.
- **AgentCore** is AWS's managed platform for running agents: Runtime hosts them and Observability shows you what they did. Those two are what this course uses; the rest of the menu is there when a problem asks for it.
- It is **framework-agnostic** — we use Strands, and the same deployment works for LangGraph, ADK, and others.
- The same `main.py` runs locally and in production, which is why we build locally first.
- Your environment is ready: credentials work, model access works, settings are saved.

**Next:** `02_build_and_test_locally.ipynb` — write the tools, assemble the agent, and watch it reason through a real AWS question on your own laptop.